In [10]:
import numpy as np
import pandas as pd
import os
import openpyxl
import re

In [11]:
data_path = os.path.join(os.getcwd(), "..", "data", "training")
raw_data_path = os.path.join(data_path, "raw")
converted_data_path = os.path.join(data_path, "converted")
os.makedirs(converted_data_path, exist_ok=True)

In [12]:
raw_files = os.listdir(raw_data_path)
raw_files = [file for file in raw_files if file.endswith(".xlsx")]
raw_files

['raw2.xlsx', 'raw.xlsx']

In [27]:
file_mapping = dict()
for file in raw_files:
    if re.match(r".*\d+.*", file.lower(), re.IGNORECASE):
        name = f"series_{re.search(r".*(\d+).*", file.lower(), flags=re.IGNORECASE).group(1)}"
    else:
        name = "series_1"
    file_mapping[file] = name
    print(f"Mapped {file} to {name}.")

file_mapping

Mapped raw2.xlsx to series_2.
Mapped raw.xlsx to series_1.


{'raw2.xlsx': 'series_2', 'raw.xlsx': 'series_1'}

In [13]:
workbooks = np.empty((len(raw_files), 2), dtype=object)
for i, file in enumerate(raw_files):
    print(f"Loading {file}...")
    workbooks[i, 0] = file
    workbooks[i, 1] = pd.ExcelFile(os.path.join(raw_data_path, file), engine="openpyxl", engine_kwargs={"read_only": True})
    print(f"Loaded {file} successfully.")

Loading raw2.xlsx...
Loaded raw2.xlsx successfully.
Loading raw.xlsx...
Loaded raw.xlsx successfully.


In [14]:
sheet_lists = [(wb[0], wb[1].sheet_names) for wb in workbooks]
sheet_lists

[('raw2.xlsx', ['Alle', 'Wurst1', 'Wurst2', 'Wurst3', 'Wurst4', 'Wurst5']),
 ('raw.xlsx',
  ['Filtered',
   '+ Annotation',
   'Messwerte Wurst 1, 1. Messreihe',
   'Messwerte Wurst 2, 1. Messreihe',
   'Messwerte Wurst 3, 1. Messreihe',
   'Messwerte Wurst 4, 1. Messreihe',
   'Messwerte Wurst 5, 1. Messreihe'])]

In [15]:
for (file, sheet_list) in sheet_lists:
    print("\n".join(sheet_list))

Alle
Wurst1
Wurst2
Wurst3
Wurst4
Wurst5
Filtered
+ Annotation
Messwerte Wurst 1, 1. Messreihe
Messwerte Wurst 2, 1. Messreihe
Messwerte Wurst 3, 1. Messreihe
Messwerte Wurst 4, 1. Messreihe
Messwerte Wurst 5, 1. Messreihe


In [28]:
sheet_mapping = dict()
for (file, sheet_list) in sheet_lists:
    for sheet in sheet_list:
        print(f"Processing {sheet} in {file}...")
        if re.match(r".*wurst.*", sheet.lower(), re.IGNORECASE):
            name = f"{file_mapping[file]}_sausage_{re.search(r".*wurst[^\d]*(\d+).*", sheet.lower(), flags=re.IGNORECASE).group(1)}"
            sheet_mapping[sheet] = name
            print(f"Mapped {sheet} to {name}.")

sheet_mapping

Processing Alle in raw2.xlsx...
Processing Wurst1 in raw2.xlsx...
Mapped Wurst1 to series_2_sausage_1.
Processing Wurst2 in raw2.xlsx...
Mapped Wurst2 to series_2_sausage_2.
Processing Wurst3 in raw2.xlsx...
Mapped Wurst3 to series_2_sausage_3.
Processing Wurst4 in raw2.xlsx...
Mapped Wurst4 to series_2_sausage_4.
Processing Wurst5 in raw2.xlsx...
Mapped Wurst5 to series_2_sausage_5.
Processing Filtered in raw.xlsx...
Processing + Annotation in raw.xlsx...
Processing Messwerte Wurst 1, 1. Messreihe in raw.xlsx...
Mapped Messwerte Wurst 1, 1. Messreihe to series_1_sausage_1.
Processing Messwerte Wurst 2, 1. Messreihe in raw.xlsx...
Mapped Messwerte Wurst 2, 1. Messreihe to series_1_sausage_2.
Processing Messwerte Wurst 3, 1. Messreihe in raw.xlsx...
Mapped Messwerte Wurst 3, 1. Messreihe to series_1_sausage_3.
Processing Messwerte Wurst 4, 1. Messreihe in raw.xlsx...
Mapped Messwerte Wurst 4, 1. Messreihe to series_1_sausage_4.
Processing Messwerte Wurst 5, 1. Messreihe in raw.xlsx...
M

{'Wurst1': 'series_2_sausage_1',
 'Wurst2': 'series_2_sausage_2',
 'Wurst3': 'series_2_sausage_3',
 'Wurst4': 'series_2_sausage_4',
 'Wurst5': 'series_2_sausage_5',
 'Messwerte Wurst 1, 1. Messreihe': 'series_1_sausage_1',
 'Messwerte Wurst 2, 1. Messreihe': 'series_1_sausage_2',
 'Messwerte Wurst 3, 1. Messreihe': 'series_1_sausage_3',
 'Messwerte Wurst 4, 1. Messreihe': 'series_1_sausage_4',
 'Messwerte Wurst 5, 1. Messreihe': 'series_1_sausage_5'}

In [31]:
for file, sheet_list in sheet_lists:
    for sheet in sheet_list:
        if sheet not in sheet_mapping:
            print(f"Skipping {sheet} in {file} as it is not mapped.")
            continue
        file_name = sheet_mapping[sheet] + ".csv"
        print(f"Processing {sheet} in {file} and saving as {file_name}...")
        df = workbooks[np.where(workbooks[:, 0] == file)[0][0], 1].parse(sheet_name=sheet)
        df.to_csv(os.path.join(converted_data_path, file_name), index=False)
        print(f"Saved {sheet} in {file} as {file_name} successfully.")

Skipping Alle in raw2.xlsx as it is not mapped.
Processing Wurst1 in raw2.xlsx and saving as series_2_sausage_1.csv...


Saved Wurst1 in raw2.xlsx as series_2_sausage_1.csv successfully.
Processing Wurst2 in raw2.xlsx and saving as series_2_sausage_2.csv...
Saved Wurst2 in raw2.xlsx as series_2_sausage_2.csv successfully.
Processing Wurst3 in raw2.xlsx and saving as series_2_sausage_3.csv...
Saved Wurst3 in raw2.xlsx as series_2_sausage_3.csv successfully.
Processing Wurst4 in raw2.xlsx and saving as series_2_sausage_4.csv...
Saved Wurst4 in raw2.xlsx as series_2_sausage_4.csv successfully.
Processing Wurst5 in raw2.xlsx and saving as series_2_sausage_5.csv...
Saved Wurst5 in raw2.xlsx as series_2_sausage_5.csv successfully.
Skipping Filtered in raw.xlsx as it is not mapped.
Skipping + Annotation in raw.xlsx as it is not mapped.
Processing Messwerte Wurst 1, 1. Messreihe in raw.xlsx and saving as series_1_sausage_1.csv...
Saved Messwerte Wurst 1, 1. Messreihe in raw.xlsx as series_1_sausage_1.csv successfully.
Processing Messwerte Wurst 2, 1. Messreihe in raw.xlsx and saving as series_1_sausage_2.csv...


In [33]:
for workbook in workbooks:
    workbook[1].close()